In [2]:
import pandas as pd
import json
import os
import glob

In [3]:
data_folder = r"C:\Research\EndpointXAI\data\Security-Datasets\datasets\atomic\windows"
all_files = glob.glob(data_folder + "/**/*.json", recursive=True)
print(f"Found {len(all_files)} files")

Found 6 files


In [4]:
all_records = []
for file in all_files:
    try:
        parts = file.replace('\\','/').split('/')
        try:
            windows_idx = parts.index('windows')
            folder_name = parts[windows_idx + 1]
        except:
            folder_name = 'unknown'
        with open(file, 'r', encoding= 'utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    record = json.loads(line)
                    record['attack_label'] = folder_name
                    all_records.append(record)
    except Exception as e:
        print(f"Skipped {file}: {e}")
df = pd.DataFrame(all_records)
print(f"Total events: {len(df)}")

Total events: 45622


In [5]:
print(df.columns.tolist())

['@timestamp', '@metadata', 'message', 'version', 'thread_id', 'user', 'beat', 'record_number', 'opcode', 'process_id', 'source_name', 'log_name', 'event_data', 'host', 'level', 'provider_guid', 'event_id', 'computer_name', 'task', 'type', 'attack_label', 'keywords', 'activity_id', 'user_data', 'DestPort', 'ProcessId', 'OpcodeValue', 'port', 'SourceAddress', 'RemoteMachineID', 'Severity', 'SourcePort', 'EventTime', 'SourceName', 'ProviderGuid', 'Channel', 'DestAddress', 'FilterRTID', 'RemoteUserID', 'EventReceivedTime', 'ExecutionProcessID', 'EventID', 'Category', 'tags', 'SeverityValue', 'Keywords', 'EventType', 'Version', 'Direction', 'SourceModuleName', 'Application', 'SourceModuleType', 'LayerName', 'RecordNumber', 'ThreadID', 'Task', 'Message', 'Hostname', 'Protocol', 'LayerRTID', '@version', 'Opcode', 'AccountName', 'AccountType', 'Image', 'TargetFilename', 'UtcTime', 'Domain', 'RuleName', 'CreationUtcTime', 'UserID', 'ProcessGuid', 'TargetObject', 'EventTypeOrignal', 'Details', 

In [6]:
important_cols = ['NewProcessName', 'ParentProcessName', 'CommandLine', 'EventID', 'SubjectUserName', 'TargetObject', 'DestinationIp', 'Channel']

for col in important_cols:
    if col in df.columns:
        print(f"Found: {col}")
    else:
        print(f"Missing: {col}")

Found: NewProcessName
Found: ParentProcessName
Found: CommandLine
Found: EventID
Found: SubjectUserName
Found: TargetObject
Found: DestinationIp
Found: Channel


In [7]:
print(df['attack_label'].value_counts())

attack_label
lateral_movement    25993
defense_evasion     17390
persistence          2239
Name: count, dtype: int64


In [8]:
df.to_csv(r"C:\Research\EndpointXAI\data\mordor_labeled.csv", index=False)
print("saved!")

saved!


In [9]:
#Load The orignal raw data
import os
import json
raw_path = r"C:\Research\EndpointXAI\data\Security-Datasets-master\datasets"
print(os.listdir(raw_path))

['atomic', 'compound']


In [10]:
print("Atomic", os.listdir(raw_path+r"\atomic"))
print("Compound:", os.listdir(raw_path + r"\compound"))

Atomic ['aws', 'linux', 'windows', '_metadata']
Compound: ['apt29', 'GoldenSAMLADFSMailAccess', 'Log4Shell', 'LSASS_campaign_01', 'LSASS_campaign_02', 'LSASS_campaign_03', 'LSASS_campaign_04', 'LSASS_campaign_05', 'LSASS_campaign_06', 'LSASS_campaign_07', 'windows', '_metadata']


In [11]:
windows_path = raw_path +r"\atomic\windows"
print(os.listdir(windows_path))

['collection', 'credential_access', 'defense_evasion', 'discovery', 'execution', 'lateral_movement', 'other', 'persistence', 'privilege_escalation']


In [12]:
cat_path = windows_path +r"\lateral_movement"
print(os.listdir(cat_path))


['host', 'network']


In [13]:
print(os.listdir(windows_path + r"\lateral_movement\host"))

['aadinternals_export_adfsdatabaseconfig_remotely.zip', 'covenant_com_wsman_automation.zip', 'covenant_copy_smb_CreateRequest.zip', 'covenant_dcom_executeexcel4macro_allowed.zip', 'covenant_dcom_executeexcel4macro_blocked.zip', 'covenant_dcom_iertutil_dll_hijack.zip', 'covenant_dcom_registerxll.zip', 'covenant_psremoting_command.zip', 'covenant_psremoting_grunt.zip', 'covenant_sc_query_dcerpc_smb_svcctl.zip', 'covenant_sharpsc_create_dcerpc_smb_svcctl.zip', 'covenant_sharpsc_query_dcerpc_smb_svcctl.zip', 'covenant_sharpsc_start_dcerpc_smb_svcctl.zip', 'covenant_sharpsc_stop_dcerpc_smb_svcctl.zip', 'covenant_sharpwmi_create_dcerpc_wmi.zip', 'covenant_wmi_remote_event_subscription_ActiveScriptEventConsumers.zip', 'covenant_wmi_wbemcomn_dll_hijack.zip', 'empire_dcom_shellwindows_stager.zip', 'empire_msbuild_dcerpc_wmi_smb.zip', 'empire_psexec_dcerpc_tcp_svcctl.zip', 'empire_psremoting_stager.zip', 'empire_shell_dcerpc_smb_service_dll_hijack.zip', 'empire_smbexec_dcerpc_smb_svcctl.zip', 'e

In [14]:
import os, json, zipfile
import pandas as pd

windows_path = r"C:\Research\EndpointXAI\data\Security-Datasets-master\datasets\atomic\windows"

# Only use the 3 categories you labeled before
target_categories = {
    'lateral_movement': 'lateral_movement',
    'defense_evasion': 'defense_evasion',
    'persistence': 'persistence'
}

records = []

for category, label in target_categories.items():
    for subfolder in ['host', 'network']:
        folder = os.path.join(windows_path, category, subfolder)
        if not os.path.exists(folder):
            continue
        for fname in os.listdir(folder):
            if not fname.endswith('.zip'):
                continue
            zip_path = os.path.join(folder, fname)
            try:
                with zipfile.ZipFile(zip_path, 'r') as z:
                    for name in z.namelist():
                        if name.endswith('.json'):
                            with z.open(name) as f:
                                for line in f:
                                    try:
                                        event = json.loads(line)
                                        records.append({
                                            'EventID': event.get('EventID'),
                                            'CommandLine': event.get('CommandLine'),
                                            'ParentProcessName': event.get('ParentProcessName'),
                                            'TargetObject': event.get('TargetObject'),
                                            'DestinationIp': event.get('DestinationIp'),
                                            'attack_label': label
                                        })
                                    except:
                                        pass
            except:
                pass

df = pd.DataFrame(records)
print(f"Extracted {len(df)} events")
print(df['attack_label'].value_counts())
df.to_csv(r"C:\Research\EndpointXAI\data\mordor_labeled.csv", index=False)
print("Saved!")

Extracted 630238 events
attack_label
defense_evasion     242239
lateral_movement    196869
persistence         191130
Name: count, dtype: int64
Saved!
